# M7.2 · Triton kernel'ları

Python'da GPU çekirdeği yazmak. Program/blok ayrıştırmasını ve maskeleri elle hesaplıyor, saf Python'da Triton'un "blok programı" modelini maskeli yükleme/saklama ile taklit ediyor, NumPy'da tiled matmul'ü program-başı bir fonksiyon olarak kuruyor, Triton'da vektör toplama ve satır softmax'ı yazıyor ve en sonda füzyonlu ReLU epilogue'lu bir matmul çekirdeği yazıyorsun. GPU yoksa çekirdekler Triton yorumlayıcısıyla CPU'da çalışır.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m7-2-triton.html](https://m-gonul.github.io/pytorch-lab/m7-2-triton.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m7_2_triton.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def ls():\n        mem = [10, 20, 30]\n        ensure(need(ns, \"load\")(mem, [0, 2, 5], [True, True, False], other=-1) == [10, 30, -1], \"maskeli load\")\n        need(ns, \"store\")(mem, [1, 7], [99, 77], [True, False])\n        ensure(mem == [10, 99, 30], \"maskeli store dizinin dışına yazmamalı\")\n    R.test(\"load/store maskeleri\", ls)\n\n    def add():\n        n = 1000\n        x, y, out = list(range(n)), [0.5] * n, [None] * n\n        need(ns, \"launch\")(need(ns, \"add_program\"), 4, x, y, out, n, 256)\n        ensure(out == [i + 0.5 for i in range(n)], \"sonuç yanlış\")\n        out2 = [None] * (n + 5)\n        ns[\"launch\"](ns[\"add_program\"], 4, x, y, out2, n, 256)\n        ensure(out2[n:] == [None] * 5, \"maskeli program n'yi aşmamalı\")\n    R.test(\"add_program\", add)\n\n    def sm():\n        rng = random.Random(0)\n        rows, cols = 5, 13\n        X = [rng.uniform(-30, -10) for _ in range(rows * cols)]\n        Y = [None] * (rows * cols)\n        need(ns, \"launch\")(need(ns, \"softmax_program\"), rows, X, Y, cols, 16)\n        for r in range(rows):\n            row = X[r * cols:(r + 1) * cols]\n            m = max(row); e = [math.exp(v - m) for v in row]\n            close(Y[r * cols:(r + 1) * cols], [v / sum(e) for v in e], what=f\"satır {r}\")\n    R.test(\"softmax_program (tamamen negatif satırlar, BLOCK > n_cols)\", sm)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"load_tile döngüsüz\", lambda: max_loops(src, 0, func=\"load_tile\"))\n    R.test(\"matmul_program: yalnız K döngüsü\", lambda: max_loops(src, 1, func=\"matmul_program\"))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    M = np.arange(20.0).reshape(4, 5)\n\n    def lt():\n        close(need(ns, \"load_tile\")(M, 2, 3, 4, 4), [[13, 14, 0, 0], [18, 19, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]])\n    R.test(\"load_tile: kenarda sıfır dolgu\", lt)\n    rng = np.random.default_rng(0)\n\n    def mm():\n        for (m, k, n), (bm, bn, bk) in (((70, 50, 40), (16, 16, 16)), ((33, 17, 9), (8, 4, 16)), ((16, 16, 16), (16, 16, 16))):\n            A, B = rng.normal(size=(m, k)), rng.normal(size=(k, n))\n            close(need(ns, \"matmul\")(A, B, bm, bn, bk), A @ B, what=f\"{m}×{k}×{n}, tile {bm}/{bn}/{bk}\")\n    R.test(\"matmul: tek sayılı şekiller\", mm)\n\n    def calls():\n        cnt = [0]\n        orig = ns[\"matmul_program\"]\n\n        def spy(*a):\n            cnt[0] += 1\n            return orig(*a)\n        ns[\"matmul_program\"] = spy\n        g = ns[\"matmul\"].__globals__\n        g[\"matmul_program\"] = spy\n        try:\n            ns[\"matmul\"](rng.normal(size=(70, 50)), rng.normal(size=(50, 40)), 16, 16, 16)\n        finally:\n            g[\"matmul_program\"] = orig\n            ns[\"matmul_program\"] = orig\n        ensure(cnt[0] == 15, f\"{cnt[0]} program; 5 × 3 = 15 bekleniyordu\")\n    R.test(\"grid: 15 program\", calls)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    dev = ns.get(\"DEVICE\", \"cpu\")\n    g = torch.Generator().manual_seed(0)\n\n    def ad():\n        for n in (1, 255, 256, 1000, 4097):\n            x, y = torch.randn(n, generator=g).to(dev), torch.randn(n, generator=g).to(dev)\n            close(need(ns, \"add\")(x, y), x + y, what=f\"n = {n}\")\n    R.test(\"add (farklı n, kenar blokları)\", ad)\n\n    def sm():\n        for shape in ((64, 100), (7, 1), (3, 257)):\n            x = (torch.randn(*shape, generator=g) * 5 - 20).to(dev)\n            close(need(ns, \"softmax\")(x), torch.softmax(x, -1), atol=1e-6, what=f\"{shape}\")\n    R.test(\"softmax (negatif satırlar, 2'nin kuvveti olmayan sütunlar)\", sm)\n\n    def strided():\n        x = torch.randn(10, 64, generator=g).to(dev)[:, :50]\n        close(need(ns, \"softmax\")(x), torch.softmax(x, -1), atol=1e-6, what=\"bitişik olmayan girdi\")\n    R.test(\"dilimlenmiş girdi\", strided)\n    if dev == \"cuda\" and \"softmax\" in ns:\n        x = torch.randn(4096, 1024, device=\"cuda\")\n        t1 = bench(lambda: ns[\"softmax\"](x), repeat=10)\n        t2 = bench(lambda: torch.softmax(x, -1), repeat=10)\n        R.table(\"4096 × 1024 softmax (GPU)\", [\"yöntem\", \"süre\"], [[\"senin Triton\", fmt_time(t1)], [\"torch.softmax\", fmt_time(t2)]])\n    else:\n        R.note(\"CPU yorumlayıcısında çalıştı: sonuçlar doğru, süreler anlamsız. Hız için Colab'da GPU çalışma zamanı kullan.\")\n",
 "e": "import torch\nfrom labcheck import close, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    dev = ns.get(\"DEVICE\", \"cpu\")\n    g = torch.Generator().manual_seed(0)\n    tol = 1e-4 if dev == \"cpu\" else 2e-2   # GPU'da tl.dot TF32 kullanabilir\n\n    def mm():\n        for m, k, n in ((70, 50, 40), (1, 33, 65), (64, 64, 64)):\n            a, b = torch.randn(m, k, generator=g).to(dev), torch.randn(k, n, generator=g).to(dev)\n            close(need(ns, \"matmul\")(a, b), a @ b, atol=tol, rtol=tol, what=f\"{m}×{k}×{n}\")\n    R.test(\"matmul: tek sayılı şekiller\", mm)\n\n    def relu():\n        a, b = torch.randn(45, 37, generator=g).to(dev), torch.randn(37, 29, generator=g).to(dev)\n        out = need(ns, \"matmul\")(a, b, \"relu\")\n        close(out, torch.relu(a @ b), atol=tol, rtol=tol)\n        ensure((out >= 0).all(), \"ReLU epilogue negatif bırakmamalı\")\n    R.test(\"ReLU epilogue\", relu)\n\n    def tr():\n        a = torch.randn(40, 30, generator=g).to(dev).t()\n        b = torch.randn(40, 20, generator=g).to(dev)\n        close(need(ns, \"matmul\")(a, b), a @ b, atol=tol, rtol=tol, what=\"transpoze A (stride'lar)\")\n    R.test(\"bitişik olmayan A (stride'larla)\", tr)\n    if dev == \"cuda\" and \"matmul\" in ns:\n        a, b = torch.randn(2048, 2048, device=\"cuda\"), torch.randn(2048, 2048, device=\"cuda\")\n        t1 = bench(lambda: ns[\"matmul\"](a, b), repeat=10)\n        t2 = bench(lambda: a @ b, repeat=10)\n        R.table(\"2048³ matmul (GPU)\", [\"yöntem\", \"süre\", \"TFLOP/s\"],\n                [[\"senin Triton\", fmt_time(t1), f\"{2 * 2048 ** 3 / t1 / 1e12:.1f}\"], [\"torch (cuBLAS)\", fmt_time(t2), f\"{2 * 2048 ** 3 / t2 / 1e12:.1f}\"]])\n    else:\n        R.note(\"CPU yorumlayıcısında çalıştı: doğruluk sınandı; hız için GPU gerekir.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m7_2_triton as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Triton'da bir **program**, verinin bir **blokunu** işler. Çekirdek, blok boyutunda vektörlerle yazılır:

```python
@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask)
    y = tl.load(y_ptr + offs, mask=mask)
    tl.store(out_ptr + offs, x + y, mask=mask)

grid = (triton.cdiv(n, BLOCK),)
add_kernel[grid](x, y, out, n, BLOCK=1024)
```

- `tl.constexpr` derleme zamanı sabiti; `tl.arange` uzunluğu 2'nin kuvveti olmalı.
- **Maske**, blok tensörü aşınca dışarıyı okumayı/yazmayı engeller; `other=` maskelenen konumlara konacak değer.
- İşaretçi aritmetiği: `ptr + offs` bir işaretçi bloğu; 2-boyutlu bloklar için `ptr + rows[:, None] * stride + cols[None, :]`.

**GPU yoksa:** `TRITON_INTERPRET=1` ortam değişkeniyle (Triton import edilmeden **önce**) çekirdekler NumPy tabanlı bir yorumlayıcıda CPU tensörleri üzerinde çalışır. Hızlı değil ama sonuçlar aynı; doğruluğu burada, hızı GPU'da (Colab) sınarsın.

## 1 · Elle

Son program $3\cdot256=768$'den başlar; $1000-768=232$ eleman geçerli. Softmax'ta maskelenen sütunlara $-\infty$ konursa `max` etkilenmez, `exp` 0 verir ve toplam bozulmaz. 0 konsaydı, tüm değerleri negatif olan bir satırda max yanlış çıkardı.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Vektör toplama, $N=1000$, `BLOCK = 256`**

- `progs`: Program sayısı (grid)
- `valid`: Son programda maskesi doğru olan eleman

**Satır softmax'ı, $64\times100$ matris, program başına bir satır**

- `block`: `BLOCK` (2'nin kuvveti, ≥ sütun sayısı)
- `pad`: Maskelenen konumlara `other=` ile ne yüklenmeli? (sayı olarak, $-\infty$ için −1 yaz)

**Matmul $C_{70\times40}=A_{70\times50}B_{50\times40}$, `BM = BN = BK = 16`**

- `grid`: Program sayısı
- `ksteps`: Program başına K-döngüsü adımı

In [ ]:
cevaplar = dict(
    progs=None,
    valid=None,
    block=None,
    pad=None,
    grid=None,
    ksteps=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"progs":[4,0],"valid":[232,0],"block":[128,0],"pad":[-1,0],"grid":[15,0],"ksteps":[4,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python ile Triton'un blok modeli. "Bellek" düz bir liste, "işaretçi" bir indeks.

- `arange(n)` → `list(range(n))`.
- `load(mem, offs, mask, other=0.0)` → liste; maskeli konumlarda `other`.
- `store(mem, offs, vals, mask)` — yalnız maskeli konumlara yaz.
- `launch(program, grid, *args)` — her `pid in range(grid)` için `program(pid, *args)`.
- `add_program(pid, x, y, out, n, BLOCK)`.
- `softmax_program(pid, X, Y, n_cols, BLOCK)` — `X`, `Y` satır-ana düz liste; satır `pid`; maskelenenlere `other=-inf`.

In [ ]:
%%writefile stage_b.py
import math


def arange(n):
    ...


def load(mem, offs, mask, other=0.0):
    ...


def store(mem, offs, vals, mask):
    ...


def launch(program, grid, *args):
    ...


def add_program(pid, x, y, out, n, BLOCK):
    ...


def softmax_program(pid, X, Y, n_cols, BLOCK):
    ...


n = 1000
x, y, out = list(range(n)), [1.0] * n, [None] * n
launch(add_program, (n + 255) // 256, x, y, out, n, 256)
print(out[:3], out[-1])
X = [-5.0, -6.0, -7.0, 1.0, 2.0, 3.0]; Y = [None] * 6
launch(softmax_program, 2, X, Y, 3, 4)
print(Y)

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy ile Triton tarzı tiled matmul. Programlar `(pid_m, pid_n)`; her program `C`'nin bir `BM × BN` tile'ını hesaplar.

- `load_tile(M, r0, c0, R, Cc)` → `(R, Cc)` tile; matrisin dışına düşen kısımlar 0 (maskeli yükleme).
- `matmul_program(A, B, C, pid_m, pid_n, BM, BN, BK)` — K boyunca `BK` adımlarla `acc += load_tile(A, …) @ load_tile(B, …)`, sonra `C`'ye maskeli yaz.
- `matmul(A, B, BM=16, BN=16, BK=16)` — grid'i başlat. Program'lar üzerinde iki seviye döngü + programın içinde K döngüsü serbest; program içinde eleman döngüsü yok.

In [ ]:
%%writefile stage_c.py
import numpy as np


def load_tile(M, r0, c0, R, Cc):
    ...


def matmul_program(A, B, C, pid_m, pid_n, BM, BN, BK):
    ...


def matmul(A, B, BM=16, BN=16, BK=16):
    ...


rng = np.random.default_rng(0)
A, B = rng.normal(size=(70, 50)), rng.normal(size=(50, 40))
print(np.abs(matmul(A, B) - A @ B).max())

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Triton (GPU yoksa yorumlayıcı). Dosyanın başı hazır: GPU yoksa `TRITON_INTERPRET=1`, `DEVICE` seçimi.

- `add_kernel` + `add(x, y)` — `BLOCK = 256`.
- `softmax_kernel(X, Y, stride, n_cols, BLOCK)` + `softmax(x)` — program başına bir satır, `BLOCK = triton.next_power_of_2(n_cols)`; satırı yükle (maskeli, `other=-inf`), max çıkar, exp, toplam, böl, sakla.

Triton paket olarak yalnız Linux x86-64'te var; macOS/Windows'ta bu aşamayı Colab'da çalıştır.

In [ ]:
%%writefile stage_d.py
import os
import torch

if not torch.cuda.is_available():
    os.environ.setdefault("TRITON_INTERPRET", "1")   # GPU yoksa CPU'da yorumlayıcı
import triton
import triton.language as tl

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    ...


def add(x, y):
    ...


@triton.jit
def softmax_kernel(X, Y, stride, n_cols, BLOCK: tl.constexpr):
    ...


def softmax(x):
    ...


x = torch.randn(1000, device=DEVICE)
print((add(x, x) - 2 * x).abs().max())

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`matmul_kernel(A, B, C, M, N, K, stride…, BM, BN, BK, ACT)` + `matmul(a, b, activation=None)`:

- 2-boyutlu grid `(cdiv(M, BM), cdiv(N, BN))`.
- `acc = tl.zeros((BM, BN), dtype=tl.float32)`; K boyunca `tl.dot(a_tile, b_tile)`; maskeli yüklemelerde `other=0`.
- `ACT: tl.constexpr` — `"relu"` ise `tl.store`'dan önce `tl.maximum(acc, 0)` (epilogue füzyonu).

Kontrol tek sayılı şekillerle doğrular. CPU yorumlayıcısında süre anlamsız; GPU'da kontrol `torch.matmul` ile karşılaştırma tablosu da basar.

In [ ]:
%%writefile stage_e.py
import os
import torch

if not torch.cuda.is_available():
    os.environ.setdefault("TRITON_INTERPRET", "1")
import triton
import triton.language as tl

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


@triton.jit
def matmul_kernel(A, B, C, M, N, K, sam, sak, sbk, sbn, scm, scn,
                  BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr, ACT: tl.constexpr):
    ...


def matmul(a, b, activation=None):
    ...


a, b = torch.randn(70, 50, device=DEVICE), torch.randn(50, 40, device=DEVICE)
print((matmul(a, b, "relu") - torch.relu(a @ b)).abs().max())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Tillet, Kung, Cox (2019): Triton

[Tillet, Kung, Cox — Triton: An Intermediate Language and Compiler for Tiled Neural Network Computations (MAPL 2019)](https://dl.acm.org/doi/10.1145/3315508.3329973)

Makalenin döşemeli (tiled) programlama modelinin karşılığı olarak Triton'da matmul çekirdeğini adım adım geliştir: blok boyutları (BLOCK_M/N/K), grup sıralı (swizzled) program kimlikleri ile L2 yeniden kullanımı, otomatik ayar (triton.autotune). Her adımda performansı cuBLAS'a (torch.matmul) göre raporla; makalenin Şekil 8'indeki gibi şekil taraması yap ve hangi şekillerde geride kaldığını açıkla.


**Adımlar**

1. CPU'da TRITON_INTERPRET=1 ile doğruluk; performans Colab GPU'sunda.

**Kabul ölçütleri**

- Doğru sonuç; büyük kare matrislerde cuBLAS'ın makul bir yüzdesi.
- Swizzle'ın L2 isabetine/performansa etkisi ölçülmüş.

## Kaynaklar

- [Triton — Tutorials (vector add, fused softmax, matrix multiplication)](https://triton-lang.org/main/getting-started/tutorials/) — MIT
- [Tillet, Kung, Cox — Triton: An Intermediate Language and Compiler for Tiled Neural Network Computations](https://www.eecs.harvard.edu/~htk/publication/2019-mapl-tillet-kung-cox.pdf) — ©
- [Stanford CS336 — Assignment 2 (Triton FlashAttention-2)](https://github.com/stanford-cs336/assignment2-systems) — MIT
- [GPU MODE — Lecture 14: Practitioners Guide to Triton](https://github.com/gpu-mode/lectures) — Apache-2.0